In [2]:
from tqec.compile.compile import compile_block_graph  # noqa: E402
from tqec.computation.block_graph import BlockGraph
from tqec.computation.correlation import (
    ConditionalCorrelationSurface,
    CorrelationSurface,
    ZXEdge,
    ZXNode,
)
from tqec.computation.cube import ConditionalLeafCubeKind
from tqec.utils.enums import Basis
from tqec.utils.position import Position3D, Direction3D
from tqec.compile import compile_block_graph

In [2]:
def build_brancing_tree(n: int) -> BlockGraph:
    """Build Blockgraph with n conditional cubes
    
    The stucture of an n=2 tree is as follows (R=red, B=blue, C=condiional):
    C  R
    |  |
    B--B  C
       |  |
       B--B
       |
       R
    The condions are always the B--B piped on the layer below the conditional cube C
    """
    if n < 1:
        raise Exception(f"The number of conditional cubes must be >1. Got {n} instead.")
        
    g = BlockGraph(f"tree_{n}")
    
    def add_branch_at_z(g: BlockGraph, z: int):
        # sign parity
        s = 1 - 2 * (z % 2)
        
        p = Position3D(0, 0, z)
        b = Position3D(s, 0, z)
        c = Position3D(s, 0, z + 1) 

        condition = CorrelationSurface(
            span=frozenset({ZXEdge(ZXNode(p, Basis.X), ZXNode(b, Basis.X))})
        )

        g.add_cube(p, "ZXZ")
        g.add_cube(b, "ZXZ")
        g.add_cube(
            c,
            ConditionalLeafCubeKind.ZXZ_ZXX,
            condition=condition
        )
    
        g.add_pipe(p, b)
        g.add_pipe(b, c)

        return p

    bottom = Position3D(0, 0, 0)
    top = Position3D(0, 0, n + 1)
    branch_positions = [add_branch_at_z(g, z) for z in range(1, n + 1)]
    
    g.add_cube(bottom, "ZXZ")
    g.add_cube(top, "ZXZ")

    g.add_pipe(bottom, branch_positions[0])
    for p1, p2 in zip(branch_positions[:-1], branch_positions[1:]):
        g.add_pipe(p1, p2) 
    g.add_pipe(branch_positions[-1], top)

    
    return g
        
        

In [4]:
g = build_brancing_tree(3)
g.view_as_html()

In [5]:
def make_stim_text(g: BlockGraph, k:int = 1):
    compiled = compile_block_graph(g, observables=None)
    stim_text = compiled.generate_conditional_stim_text(k=k)
    return stim_text
    
stim_text = make_stim_text(g)
#print(stim_circuit)

/Users/isaac/Documents/entropica/tqec_ccf/src/tqec/compile/compile.py:336: UserWarning: BlockGraph contains conditional cubes; skipping correlation-surface validity check (pyzx ZX conversion does not support ConditionalLeafCubeKind).
  ): compile_correlation_surface_to_abstract_observable(


In [8]:
for n in map(lambda i: 2**i, range(8)):
    print(f"Generating stim text for n={n}")
    
    g_tree = build_brancing_tree(n)
    text = make_stim_text(g_tree)    

    with open(f"output/branch_{n}.stim", "w") as f:
        f.write(text)
        print(f"Wrote to {f.name}")

Generating stim text for n=1
Wrote to output/branch_1.stim
Generating stim text for n=2
Wrote to output/branch_2.stim
Generating stim text for n=4
Wrote to output/branch_4.stim
Generating stim text for n=8
Wrote to output/branch_8.stim
Generating stim text for n=16
Wrote to output/branch_16.stim
Generating stim text for n=32
Wrote to output/branch_32.stim
Generating stim text for n=64
Wrote to output/branch_64.stim
Generating stim text for n=128
Wrote to output/branch_128.stim
